### Librerias


In [3]:
#Importar las librerías necesarias para la adquisición y manipulación de datos.
import pandas as pd      #Libreria de manipulación de datos
import numpy as np       #Libreria de cálculo numérico
import pdfplumber        #Libreria para extraer texto de archivos PDF
import re                #Libreria para expresiones regulares
import pathlib           #Libreria para manipulación de rutas de archivos
from pathlib import Path # Manejo moderno de rutas de archivos

print(f"Versión de pandas: {pd.__version__}")
print(f"Versión de numpy: {np.__version__}")
print(f"Versión de pdfplumber: {pdfplumber.__version__}")
print(f"Versión de re: {re.__version__}")


Versión de pandas: 3.0.1
Versión de numpy: 2.4.3
Versión de pdfplumber: 0.11.9
Versión de re: 2.2.1


### Archivo raw CSV - Resultados - Liga Promesa, Sub 15, TA 2025, Grupo B


#### Carga de Archivo

In [27]:
#Carga de Archivo CSV con resultados de la Liga Promesas Caroní - Sub15 - Torneo Apertura 2025 - Grupo B.
df_Resultados_TA25_GrupoB = pd.read_csv("../datos/raw/Resultados_TA25_GrupoB_raw.csv")

#Se verifica que el dataset se haya cargado correctamente, mostrando su forma (número de filas y columnas).                                                                   
print("Datasets cargados correctamente:")
print(f"Resultados - Liga Promesas, Mun. Caroní - Sub15 - TA2025 - Grupo B, tiene: {df_Resultados_TA25_GrupoB.shape[0]:>3} filas x{df_Resultados_TA25_GrupoB.shape[1]:>3} columnas")

Datasets cargados correctamente:
Resultados - Liga Promesas, Mun. Caroní - Sub15 - TA2025 - Grupo B, tiene:  42 filas x  6 columnas


#### Exploración Inicial

In [29]:
print("Primeras filas del dataset: Resultados - Liga Promesas, Mun. Caroní - Sub15 - TA2025 - Grupo B")
display(df_Resultados_TA25_GrupoB.head())

Primeras filas del dataset: Resultados - Liga Promesas, Mun. Caroní - Sub15 - TA2025 - Grupo B


,Jornada,Fecha/hora,Estadio,Partido,Resultado,Estado
0,1,12.04.2025 09:30,LICEO LOS OLIVOS,ACADEMIA M.f.c - CORE SPORT,0:4,JUGADO
1,1,10.04.2025 15:30,NaN,ACADEMIA DEPORTIVA EMANUEL - ESCUELA DE FUTBOL...,3:3,JUGADO
2,1,11.04.2025 16:00,''LOS OLIVOS'' PTO. ORDAZ,LOS OLIVOS - ACADEMIA SUR AEROPUERTO fc,0:5,JUGADO
3,2,21.03.2025 15:00,CANCHA VILLA BETANIA,S LUGO fc - ACADEMIA M.f.c,1:2,JUGADO
4,2,22.03.2025 14:00,CANCHA EL CAMPITO-CORE 8,CORE SPORT - ACADEMIA DEPORTIVA EMANUEL,3:0*,JUGADO


In [30]:
#Información general del dataset, para conocer los tipos de datos, número de valores no nulos y memoria utilizada.
df_Resultados_TA25_GrupoB.info()

<class 'pandas.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Jornada     42 non-null     int64
 1   Fecha/hora  42 non-null     str  
 2   Estadio     28 non-null     str  
 3   Partido     42 non-null     str  
 4   Resultado   42 non-null     str  
 5   Estado      42 non-null     str  
dtypes: int64(1), str(5)
memory usage: 2.1 KB


In [31]:
#Se muestra la cantidad de Jornadas de la Temporada.
print(" Cantidad de Jornadas Jugadas en Liga Promesas, Mun. Caroní - Sub15 - TA2025 - Grupo B: ", df_Resultados_TA25_GrupoB["Jornada"].nunique())

 Cantidad de Jornadas Jugadas en Liga Promesas, Mun. Caroní - Sub15 - TA2025 - Grupo B:  14


#### Revisión y Corrección de Valores Nulos

In [32]:
#Se revisan los valores nulos por columna, para identificar posibles datos faltantes.
print("Valores Nulos por Columna:")
Valores_Nulos= df_Resultados_TA25_GrupoB.isnull().sum()
print(Valores_Nulos)

Valores Nulos por Columna:
Jornada        0
Fecha/hora     0
Estadio       14
Partido        0
Resultado      0
Estado         0
dtype: int64


In [33]:
#Se copia el dataframe, para realizar las transformaciones necesarias sin afectar el original.
df_Resultados = df_Resultados_TA25_GrupoB.copy()

#Se Rellenan los valores nulos de la columna "Estadio" con "DESCONOCIDO"
df_Resultados['Estadio'] = df_Resultados['Estadio'].fillna("DESCONOCIDO")

#Se verifica que no existan valores nulos en el dataframe después de la transformación.
print("Valores Nulos por Columna:")
Valores_Nulos= df_Resultados.isnull().sum()
print(Valores_Nulos)

#Se muestran las primeras filas del dataset, para verificar su contenido.
display(df_Resultados.head())

Valores Nulos por Columna:
Jornada       0
Fecha/hora    0
Estadio       0
Partido       0
Resultado     0
Estado        0
dtype: int64


,Jornada,Fecha/hora,Estadio,Partido,Resultado,Estado
0,1,12.04.2025 09:30,LICEO LOS OLIVOS,ACADEMIA M.f.c - CORE SPORT,0:4,JUGADO
1,1,10.04.2025 15:30,DESCONOCIDO,ACADEMIA DEPORTIVA EMANUEL - ESCUELA DE FUTBOL...,3:3,JUGADO
2,1,11.04.2025 16:00,''LOS OLIVOS'' PTO. ORDAZ,LOS OLIVOS - ACADEMIA SUR AEROPUERTO fc,0:5,JUGADO
3,2,21.03.2025 15:00,CANCHA VILLA BETANIA,S LUGO fc - ACADEMIA M.f.c,1:2,JUGADO
4,2,22.03.2025 14:00,CANCHA EL CAMPITO-CORE 8,CORE SPORT - ACADEMIA DEPORTIVA EMANUEL,3:0*,JUGADO


#### Normalizar Nombres de Equipos, en Mayusculas

In [34]:
#Se convierten los nombres de los equipos en mayúsculas, para estandarizar la información
df_Resultados['Partido'] = df_Resultados['Partido'].str.upper()

#Se muestran las primeras filas del dataset, para verificar su contenido.
display(df_Resultados.head())

,Jornada,Fecha/hora,Estadio,Partido,Resultado,Estado
0,1,12.04.2025 09:30,LICEO LOS OLIVOS,ACADEMIA M.F.C - CORE SPORT,0:4,JUGADO
1,1,10.04.2025 15:30,DESCONOCIDO,ACADEMIA DEPORTIVA EMANUEL - ESCUELA DE FUTBOL...,3:3,JUGADO
2,1,11.04.2025 16:00,''LOS OLIVOS'' PTO. ORDAZ,LOS OLIVOS - ACADEMIA SUR AEROPUERTO FC,0:5,JUGADO
3,2,21.03.2025 15:00,CANCHA VILLA BETANIA,S LUGO FC - ACADEMIA M.F.C,1:2,JUGADO
4,2,22.03.2025 14:00,CANCHA EL CAMPITO-CORE 8,CORE SPORT - ACADEMIA DEPORTIVA EMANUEL,3:0*,JUGADO


#### Corregir Tipos de Datos

In [35]:
#Verificar tipo de Datos en Campo Fecha/hora
print("Tipo de Datos en Campo fecha/hora: ", df_Resultados['Fecha/hora'].dtype)

#Convertir Campo Fecha/hora de String a DateTime.
df_Resultados['Fecha/hora'] = pd.to_datetime(df_Resultados['Fecha/hora'], format='%d.%m.%Y %H:%M')
print("Tipo de Datos en Campo fecha/hora, luego de la conversión: ", df_Resultados['Fecha/hora'].dtype)

#Se muestran las primeras filas del dataset, para verificar su contenido.
display(df_Resultados.head())

#Información general del dataset, para verificar las correcciones realizadas.
df_Resultados.info()

Tipo de Datos en Campo fecha/hora:  str
Tipo de Datos en Campo fecha/hora, luego de la conversión:  datetime64[us]


,Jornada,Fecha/hora,Estadio,Partido,Resultado,Estado
0,1,2025-04-12 09:30:00,LICEO LOS OLIVOS,ACADEMIA M.F.C - CORE SPORT,0:4,JUGADO
1,1,2025-04-10 15:30:00,DESCONOCIDO,ACADEMIA DEPORTIVA EMANUEL - ESCUELA DE FUTBOL...,3:3,JUGADO
2,1,2025-04-11 16:00:00,''LOS OLIVOS'' PTO. ORDAZ,LOS OLIVOS - ACADEMIA SUR AEROPUERTO FC,0:5,JUGADO
3,2,2025-03-21 15:00:00,CANCHA VILLA BETANIA,S LUGO FC - ACADEMIA M.F.C,1:2,JUGADO
4,2,2025-03-22 14:00:00,CANCHA EL CAMPITO-CORE 8,CORE SPORT - ACADEMIA DEPORTIVA EMANUEL,3:0*,JUGADO


<class 'pandas.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Jornada     42 non-null     int64         
 1   Fecha/hora  42 non-null     datetime64[us]
 2   Estadio     42 non-null     str           
 3   Partido     42 non-null     str           
 4   Resultado   42 non-null     str           
 5   Estado      42 non-null     str           
dtypes: datetime64[us](1), int64(1), str(4)
memory usage: 2.1 KB


#### Exportar Datos procesados a nuevo archivo CSV

In [36]:
#Exportar el dataset corregido a un nuevo archivo CSV, para su uso en análisis posteriores.
df_Resultados.to_csv("../datos/processed/Resultados_TA25_GrupoB_clean.csv", index=False)

print("Dataset corregido se ha exportado correctamente")
print(f"Resultados - Liga Promesas, Mun. Caroní - Sub15 - TA2025 - Grupo B, corregido, tiene: {df_Resultados.shape[0]:>3} filas x{df_Resultados.shape[1]:>3} columnas")

Dataset corregido se ha exportado correctamente
Resultados - Liga Promesas, Mun. Caroní - Sub15 - TA2025 - Grupo B, corregido, tiene:  42 filas x  6 columnas


### Archivo raw PDF - Jornada 10 - Liga Promesas, Sub15, TA 2025, Grupo B

#### Carga de Archivo

In [ ]:
#Carga de Archivo PDF con jornada de la Liga Promesas Caroní - Sub15 - Torneo Apertura 2025 - Grupo B.

ruta_pdf = pathlib.Path("../datos/raw/TA-2025/TA-F10-01.05.25.pdf") #Definir la ruta del archivo PDF a procesar.

with pdfplumber.open(ruta_pdf) as pdf:
      
    alineacion=pdf.pages[0].extract_text() #Extraer el texto de la primera página del PDF, que contiene las alineaciones de los equipos.
    incidencias=pdf.pages[1].extract_text() #Extraer el texto de la segunda página del PDF, que contiene las incidencias del partido.
    
#Concatenar el texto de ambas páginas para tener toda la información del partido en una sola variable.
partido = alineacion + "\n" + incidencias 

#Mostrar el texto bruto extraído y concatenado. 
print("Texto bruto del PDF de la Jornada:")
print(partido)

Texto bruto del PDF de la Jornada:
COMET - Federación Venezolana de Fútbol Fecha: 25.03.2026 Hora: 21:36:32 VET Impreso por: Ailid villareal (178732977)
LIGA PROMESAS FÚTBOL CAMPO MUN CARONÍ SUB-15 2025
TA 2025 - GRUPO B
ACADEMIA DEPORTIVA EMANUEL vs ACADEMIA M.F.C
0:1 (0:1)
Publico: INFORME DE PARTIDO 28
Fecha: 01/05/2025 Hora: 11:15 Estadio: Paratepuy (Puerto Ordaz, Venezuela)
OFICIALES DE PARTIDO
Árbitro: Milano Ospedales, Rafael Jesus (CARONI)
ACADEMIA DEPORTIVA EMANUEL
Equipo A Equipo B ACADEMIA M.F.C (Venezuela)
(Venezuela)
Jugadores Jugadores
No Apellido Nombre T/S A/C No Apellido Nombre T/S A/C
1 23 Mocco Espinoza Mathias Ezer T A 99 Rosales Gomez Moises Sebastian T A 1
2 4 Parra Fazio Ricardo Javier T 5 Maita Salas Jeremias Jatniel T C 2
3 6 Figuera Diaz Maykol Jose T C 7 Adedigba Zapata Emmanuel Aderemi T 3
Oluwaseun
4 12 Rivas Cotua Luis Rafael T
8 Rincones Ruiz Guillermo Alexander T 4
5 30 Gomez Sanchez Abram David T
10 Sandoval Noguera Dionner Alejandro T 5
6 36 Bautista A

#### Extracción de Datos Estructurados

In [4]:
#Extraemos los datos del texto y devolvemos un diccionario con la información estructurada del partido. 

#################################
#EXTRACCIÓN DE DATOS DEL PARTIDO

def extraer_datos_partido(ruta_pdf):
    
    #Extraer el texto de la planilla PDF generada por COMET, utilizando pdfplumber.
    with pdfplumber.open(ruta_pdf) as pdf:
        texto=pdf.pages[0].extract_text() #Extraer el texto de la primera página del PDF, que contiene el encabezado de la planilla.
        
    resultado = {} #Crear un Diccionario vacío para almacenar los datos.
    
    #Capturar la liga
    match_liga = re.search(r'LIGA\s+(.*?)\s+FÚTBOL CAMPO', texto)
    if match_liga:
        resultado['liga'] = match_liga.group(1)
    
    #Capturar el municipio
    match_municipio = re.search(r'MUN\s+(.*?)\s+SUB', texto)
    if match_municipio:
        resultado['municipio'] = match_municipio.group(1)
    
    #Capturar la categoría        
    match_categoria = re.search(r'(SUB-\d+)', texto)
    if match_categoria:
        resultado['categoria'] = match_categoria.group(1)
    
    #Capturar el torneo    
    match_torneo = re.search(r'([A-Z]{2}\s+\d{4})\s*-', texto)
    if match_torneo:
        resultado['torneo'] = match_torneo.group(1).strip()
    
    #Capturar el grupo    
    match_grupo = re.search(r'GRUPO\s+([A-Z])', texto)
    if match_grupo:
        resultado['grupo'] = match_grupo.group(1).strip()
            
    #Capturar los nombres de los equipos
    #Se usa el "vs" como delimitador para separar el nombre del equipo local y del visitante
    #Se usa re.IGNORECASE para que detecte "vs", "VS" o "Vs"
    match_equipos = re.search(r'(.+?)\s+vs\.?\s+(.+)', texto, re.IGNORECASE)
    if match_equipos:
        # Desempaquetar los 2 grupos capturados y almacenarlos en el diccionario resultado.
        resultado['equipo_local'] = match_equipos.group(1).strip()
        resultado['equipo_visitante'] = match_equipos.group(2).strip()
      
    #Capturar el resultado del partido    
    #Buscar patrones "X:Y (X:Y)" para extraer el resultado final y el resultado del primer tiempo.
    match_resultado = re.search(r'(\d+):(\d+)\s*\((\d+):(\d+)\)', texto)
    if match_resultado:
        # Desempaquetar los 4 grupos capturados y almacenarlos en el diccionario resultado.
        resultado['goles_local'] = int(match_resultado.group(1))
        resultado['goles_visitante'] = int(match_resultado.group(2))
        resultado['goles_1T_local'] = int(match_resultado.group(3))
        resultado['goles_1T_visitante'] = int(match_resultado.group(4))
            
    #Capturar la fecha del partido
    match_fecha = re.search(r'Fecha:\s*(\d{2})/(\d{2})/(\d{4})', texto)
    if match_fecha:
        resultado['fecha'] = match_fecha.group(1) + '/' + match_fecha.group(2) + '/' + match_fecha.group(3)
  
    #Capturar la hora del partido, asegurando que no se capture un formato de hora con segundos (HH:MM:SS)
    match_hora = re.search(r'Hora:\s*(\d{2}:\d{2})(?!\s*:\d{2})', texto)
    if match_hora:
        resultado['hora'] = match_hora.group(1).strip()
    
    #Capturar el estadio, ciudad y país donde se juega, usando un patrón que busque "Estadio: Nombre (Ciudad, País)"
    match_ubicacion = re.search(r'Estadio:\s*([^(]+)\s*\(([^,]+),\s*([^)]+)\)', texto)
    if match_ubicacion:
       resultado['estadio'] = match_ubicacion.group(1).strip()
       resultado['ciudad']  = match_ubicacion.group(2).strip()
       resultado['pais']    = match_ubicacion.group(3).strip()
    else:
    # Si no se encuentra el patrón completo, colocar "Estadio: Desconocido (Ciudad Guayana, Venezuela)"
       resultado['estadio'] = "Desconocido"
       resultado['ciudad'] = "Ciudad Guayana"
       resultado['pais'] = "Venezuela"
        
    return resultado


##############################################
#EXTRACCION DE ALINEACION DEL EQUIPO A (LOCAL)

def extraer_equipo_a(ruta_pdf, nombre_equipo):
    
    # Extraer el texto del PDF generado por COMET
    with pdfplumber.open(ruta_pdf) as pdf:
        texto = pdf.pages[0].extract_text() 
   
    # Limpiar el texto basura (comillas, caracteres de tabla, etc.) para facilitar la extracción con expresiones regulares"
    texto_limpio = texto.replace('"', '').replace(',', ' ')
    
        
    # EXPRESIÓN REGULAR ESTRICTA PARA EL EQUIPO A - LOCAL
    # (\d+)                         -> Numero Lista (Obligatorio al inicio)
    # \s+                           -> Espacio
    # (\d+)                         -> Dorsal (Obligatorio tras el Numero)
    # \s+                           -> Espacio
    # ([A-Za-z\sñÑáéíóúÁÉÍÓÚ]+)    -> Nombre (Solo letras y espacios, búsqueda perezosa)
    # \s+                           -> Espacio
    # ([TtSs])                      -> Titular o Suplente (Obligatorio)
    # (?:\s+([AaCc][\d]*))?         -> Arquero o Capitan (Opcional)
   
    # Patrón de descubrimiento: (Numero Lista) (Dorsal) (Nombre) (T/S) (A/C opcional) 
    patron_descubrimiento = r"(\d+)\s+(\d+)?\s+([A-Za-z\sñÑáéíóúÁÉÍÓÚ]+)\s+([TtSs])(?:\s+([AaCc][\d]*))?"
    
    # Array que contendrá los diccionarios de cada jugador
    alineacion_local = []

    for match in re.finditer(patron_descubrimiento, texto_limpio):
        partes = match.groups()
        
        # Procesar el nombre (estricto: 4 palabras máximo)
        nombre_completo = partes[2].split()
        if len(nombre_completo) >= 4:
            nombre_final = " ".join(nombre_completo[:4])
        else:
            continue # Se ignora por formato incompleto

        # Limpiar campo A/C 
        ac_raw = partes[4] if partes[4] else ""
        ac_limpio = re.sub(r'[^AaCc]', '', ac_raw)
        
        # Construir el diccionario para el registro actual
        jugador_dict = {
            "equipo": nombre_equipo, # Usamos el nombre del equipo extraído del encabezado
            "numero_lista": int(partes[0]) if partes[0] else None,
            "dorsal": int(partes[1]) if partes[1] else None,
            "nombre": nombre_final,
            "titular_suplente": partes[3].upper(), # 'T' o 'S' en mayúscula para estandarizar
            "arquero_capitan": ac_limpio.upper() if ac_limpio else "" # 'A' o 'C'
        }
        
        alineacion_local.append(jugador_dict)
        
    return alineacion_local

##############################################
#EXTRACCION DE ALINEACION DEL EQUIPO B (VISITANTE)

def extraer_equipo_b(ruta_pdf, nombre_equipo):
    # Extraer el texto del PDF generado por COMET
    with pdfplumber.open(ruta_pdf) as pdf:
        pagina = pdf.pages[0]  
        
        # 1. DIVIDIR LA PÁGINA A LA MITAD
        ancho_mitad = pagina.width / 2
        alto = pagina.height
        
        caja_visitante = (ancho_mitad, 0, pagina.width, alto)
        texto_visitante = pagina.crop(caja_visitante).extract_text()

    # 1. Limpieza inicial: Aplanamos todo en una sola línea continua
    texto_limpio = texto_visitante.replace('"', '').replace(',', ' ')
    texto_plano = re.sub(r'\s+', ' ', texto_limpio).strip()
  
    # 2. PATRÓN DE DESCUBRIMIENTO    
    # EXPRESIÓN REGULAR ESTRICTA PARA EL EQUIPO B - VISITANTE
    # (\d+)                         -> Dorsal (Obligatorio al inicio)
    # \s+                           -> Espacio
    # ([A-Za-z\sñÑáéíóúÁÉÍÓÚ]+?)    -> Nombre (Solo letras y espacios, búsqueda perezosa)
    # \s+                           -> Espacio
    # ([TtSs])                      -> Titular o Suplente (Obligatorio)
    # (?:\s+([AaCc][\d]*))?         -> Arquero o Capitan (Opcional)
    # \s+                           -> Espacio
    # (\d+)                         -> Numero Lista (Obligatorio al final)
        
    # Captura estrictamente el patrón: (Dorsal) (Nombre) (T/S) (A/C opcional) (Numero Lista)
    patron_descubrimiento = r"(\d+)\s+([A-Za-z\sñÑáéíóúÁÉÍÓÚ]+?)\s+([TtSs])(?:\s+([AaCc][\d]*))?\s+(\d+)"
    
    matches = list(re.finditer(patron_descubrimiento, texto_plano))
    alineacion_visitante = []

    for i, match in enumerate(matches):
        partes = match.groups()
        
        # Asignación correcta de índices según los grupos del Regex
        dorsal = partes[0]
        nombre_base = partes[1].strip()
        ts = partes[2].upper()
        ac_raw = partes[3] if partes[3] else ""
        lista = partes[4]
        
        ac_limpio = re.sub(r'[^AaCc]', '', ac_raw)
        
        # --- RESCATE DE NOMBRES HUÉRFANOS (FIX PARA NOMBRES LARGOS) ---
        # Calculamos dónde termina el jugador anterior y dónde empieza el siguiente
        inicio_actual = match.start()
        fin_actual = match.end()
        
        fin_anterior = matches[i-1].end() if i > 0 else 0
        inicio_siguiente = matches[i+1].start() if i < len(matches) - 1 else len(texto_plano)
        
        # Extraemos el texto suelto alrededor del núcleo de este jugador
        texto_antes = texto_plano[fin_anterior:inicio_actual].strip()
        texto_despues = texto_plano[fin_actual:inicio_siguiente].strip()
        
        # Filtramos para asegurar que solo recogemos letras (evitando basura numérica)
        texto_antes = re.sub(r'[^A-Za-zñÑáéíóúÁÉÍÓÚ\s]', '', texto_antes).strip()
        texto_despues = re.sub(r'[^A-Za-zñÑáéíóúÁÉÍÓÚ\s]', '', texto_despues).strip()
        
        # Armamos el nombre base y luego intentamos absorber palabras huérfanas si el nombre es corto
        palabras_nombre = nombre_base.split()
        
        # LÓGICA DE ABSORCIÓN: Solo suma huérfanos si le faltan palabras al nombre
        if len(palabras_nombre) < 4:
            if texto_antes:
                palabras_nombre.extend(texto_antes.split())
            # Si tras sumar el texto anterior aún le faltan palabras, sumamos el posterior
            if texto_despues and len(palabras_nombre) < 4:
                palabras_nombre.extend(texto_despues.split())
                
        # Estrictamente 4 palabras máximo
        nombre_final = " ".join(palabras_nombre[:4])
        
        # 3. Construir el diccionario para el registro actual
        jugador_dict = {
            "equipo": nombre_equipo, # Usamos el nombre del equipo extraído del encabezado
            "numero_lista": int(lista) if lista else None,
            "dorsal": int(dorsal) if dorsal else None,
            "nombre": nombre_final,
            "titular_suplente": ts,
            "arquero_capitan": ac_limpio.upper() if ac_limpio else ""
        }
        
        alineacion_visitante.append(jugador_dict)
        
    return alineacion_visitante


################################
#EXTRACCION DE LAS SUSTITUCIONES

def extraer_sustituciones(ruta_pdf):
    
    with pdfplumber.open(ruta_pdf) as pdf:
        texto=pdf.pages[1].extract_text()
    
    # 1. Aislar solo el bloque de sustituciones (Entre Equipo A y GOLES)
    match_bloque = re.search(r'Equipo A(.*?)GOLES', texto, re.DOTALL)
    if not match_bloque:
        return "No se encontró el bloque de sustituciones."
    bloque = match_bloque.group(1)

    # 2. Limpiar encabezados basura y aplanar el texto a una sola línea
    bloque = re.sub(r'Tiempo\s+No\s+ENTRA\s+No\s+SALE\s+Equipo\s+[AB]', ' ', bloque, flags=re.IGNORECASE)
    bloque = bloque.replace('\n', ' ')
    bloque = re.sub(r'\s+', ' ', bloque).strip()

    # 3. Encontrar todos los anclas de inicio (Tiempo y Dor_In)
    matches = list(re.finditer(r'\b(\d+)\s+(\d+)\b', bloque))
    N = len(matches)
    records = []

    for i in range(N):
        tiempo = matches[i].group(1)
        dor_in = matches[i].group(2)
        
        # Recortar el segmento correspondiente a esta sustitución
        start = matches[i].end()
        end = matches[i+1].start() if i < N-1 else len(bloque)
        segmento = bloque[start:end].strip()

        # Encontrar el Dorsal Saliente (el primer número suelto en el segmento)
        dor_out_match = re.search(r'\b(\d+)\b', segmento)
        if not dor_out_match:
            continue
        dor_out = dor_out_match.group(1)

        # Extraer Nombre Entrante (hasta 4 palabras antes del dor_out)
        name_in_words = segmento[:dor_out_match.start()].split()
        name_in = " ".join(name_in_words[:4])

        # Analizar el resto del segmento para ubicar el Equipo y el Nombre Saliente
        rest = segmento[dor_out_match.end():]
        
        # Buscar el Nombre del Equipo (Palabras en mayúsculas/puntos al final)
        team_matches = list(re.finditer(r'\b(?:[A-Z0-9.]{2,}(?:\s+[A-Z0-9.]{2,})*)\b', rest))
        if team_matches:
            team_match = team_matches[-1] # Tomar la última coincidencia (el equipo)
            team = team_match.group(0).strip()
            mid_words = rest[:team_match.start()].split()
            
            # Lo que queda DESPUÉS del equipo son nombres desbordados por el salto de línea
            pre_leaked_next = rest[team_match.end():].split()
        else:
            team = "EQUIPO_DESCONOCIDO"
            mid_words = rest.split()
            pre_leaked_next = []

        records.append({
            'tiempo': tiempo,
            'dor_in': dor_in,
            'name_in': name_in,
            'dor_out': dor_out,
            'mid_words': mid_words,
            'team': team,
            'pre_leaked_next': pre_leaked_next
        })

    # 4. Reconstrucción Inteligente (Stitching)
    sustituciones_finales = []
    
    for i in range(len(records)):
        rec = records[i]
        out_words = rec['mid_words'][:]

        # Si a la línea principal le faltan palabras, pedimos prestado del fragmento 
        # que se filtró en el renglón anterior (guardado en la sustitución pasada)
        if len(out_words) < 4 and i > 0:
            out_words.extend(records[i-1]['pre_leaked_next'])

        # Asegurarnos de tomar exactamente 4 componentes (2 apellidos + 2 nombres)
        name_out = " ".join(out_words[:4])
        
        # Ensamblaje final estructurado
        sustitucion_dict = {
            "equipo": rec['team'],
            "minuto": int(rec['tiempo']),
            "dorsal_entra": int(rec['dor_in']),
            "jugador_entra": rec['name_in'],
            "dorsal_sale": int(rec['dor_out']),
            "jugador_sale": name_out            
        }
        
        sustituciones_finales.append(sustitucion_dict)

    return sustituciones_finales


#############################
#EXTRACCION DE LOS GOLEADORES

def extraer_goles(ruta_pdf):
    goles_finales = []
    
    with pdfplumber.open(ruta_pdf) as pdf:
        # Página 1: Información general y nombres de equipos
        # Página 2: Eventos y tabla de goles
        pag_info = pdf.pages[0]
        pag_goles = pdf.pages[1]
        
        # --- 1. Extracción automática de nombres de equipos ---
        texto_encabezado = pag_info.extract_text() or ""
        match_equipos = re.search(r'(.+?)\s+vs\.?\s+(.+)', texto_encabezado, re.IGNORECASE)
        
        if match_equipos:
            equipo_local = match_equipos.group(1).strip()
            equipo_visitante = match_equipos.group(2).strip()
        else:
            equipo_local, equipo_visitante = "LOCAL", "VISITANTE"

        # --- 2. Definir áreas de recorte (Cajas) ---
        palabras = pag_goles.extract_words()
        y_inicio, y_fin = 0, pag_goles.height
        
        for word in palabras:
            if word['text'].upper() == 'GOLES':
                y_inicio = word['bottom']
            elif word['text'].upper() in ['PEN:', 'AG:', 'AUTOGOL']:
                y_fin = word['top'] - 2
                break

        ancho_mitad = pag_goles.width / 2
        cajas = {
            "LOCAL": (0, y_inicio, ancho_mitad, y_fin),
            "VISITANTE": (ancho_mitad, y_inicio, pag_goles.width, y_fin)
        }

        def es_linea_nombre_valida(linea):
            l_up = linea.upper()
            prohibidas = ['PEN', 'AG', 'LES', 'GOL', 'VENEZUELA', '(']
            if any(x in l_up for x in prohibidas): return False
            if l_up.replace('.', '').strip() in equipo_local.upper().replace('.', ''): return False
            if l_up.replace('.', '').strip() in equipo_visitante.upper().replace('.', ''): return False
            return True

        # --- 3. Procesar cada columna por separado ---
        for etiqueta, coord in cajas.items():
            texto = pag_goles.crop(coord).extract_text()
            if not texto: continue
            
            bloque = re.sub(r'[ \t]+', ' ', texto).strip()
            matches = list(re.finditer(r"(\d+)'\s+(\d+)", bloque))
            nombre_equipo = equipo_local if etiqueta == "LOCAL" else equipo_visitante
            
            for i, m in enumerate(matches):
                minuto, dorsal = m.groups()
                start_actual = m.start()
                end_actual = m.end()

                seg_pre = bloque[:start_actual] if i == 0 else bloque[matches[i-1].end():start_actual]
                seg_post = bloque[end_actual:] if i == len(matches)-1 else bloque[end_actual:matches[i+1].start()]

                lineas_pre = [l.strip() for l in seg_pre.split('\n') if l.strip()]
                pre_name = ""
                
                if i == 0:
                    lineas_validas_pre = [l for l in lineas_pre if es_linea_nombre_valida(l)]
                    if lineas_validas_pre: pre_name = lineas_validas_pre[-1]
                else:
                    if len(lineas_pre) >= 1:
                        palabras_inline_prev = lineas_pre[0].split()
                        extras_pre = [l for l in lineas_pre[1:] if l.upper() not in ['PEN', 'AG', 'LES']]
                        if len(extras_pre) == 1:
                            if len(palabras_inline_prev) >= 3: pre_name = extras_pre[0]
                        elif len(extras_pre) >= 2:
                            pre_name = extras_pre[-1]

                lineas_post = [l.strip() for l in seg_post.split('\n') if l.strip()]
                inline_name = ""
                wrap_down = ""

                if lineas_post:
                    inline_name = lineas_post[0]
                    palabras_inline = inline_name.split()
                    extras_post = [l for l in lineas_post[1:] if l.upper() not in ['PEN', 'AG', 'LES']]
                    if extras_post:
                        if len(extras_post) == 1:
                            if len(palabras_inline) < 3: wrap_down = extras_post[0]
                        elif len(extras_post) >= 2:
                            wrap_down = extras_post[0]

                nombre_crudo = f"{pre_name} {inline_name} {wrap_down}".strip()
                
                # Ajuste de formato (Apellidos Nombres)
                nombre_crudo = re.sub(
                    r'(Emmanuel\s+Aderemi)\s+(Adedigba\s+Zapata)', 
                    r'\2 \1', 
                    nombre_crudo, 
                    flags=re.IGNORECASE
                )
                
                todas_las_palabras = re.findall(r'[a-zA-ZñÑáéíóúÁÉÍÓÚ]+', nombre_crudo)
                nombre_limpio = [w for w in todas_las_palabras if w.upper() not in ['PEN', 'AG', 'LES']]
                nombre_final = " ".join(nombre_limpio[:4])

                tipo_match = re.search(r'\b(PEN|AG)\b(?!:)', seg_post.upper())
                tipo_gol = tipo_match.group(1).upper() if tipo_match else "JUGADA"
                
                gol_dict = {
                    "equipo": nombre_equipo,
                    "minuto": int(minuto),
                    "dorsal": int(dorsal),
                    "jugador": nombre_final,
                    "tipo_gol": tipo_gol,
                    "condicion": etiqueta # Almacena si fue 'LOCAL' o 'VISITANTE'
                }
                
                goles_finales.append(gol_dict)
                
    return goles_finales
      

#############################################################################
#EXTRACCION DE LOS AMONESTADOS (TARJETA AMARILLA) Y EXPULSADOS (TARJETA ROJA)

def extraer_tarjetas(ruta_pdf):
    resultados_disciplina = []

    with pdfplumber.open(ruta_pdf) as pdf:
        pagina = pdf.pages[1]  # Usualmente la página 2 en reportes COMET

        # 1. BUSCAR COORDENADAS (Y) DE LAS ANCLAS
        y_amarillas = 0
        y_rojas = 0
        y_tecnico = pagina.height

        for linea in pagina.extract_text_lines():
            texto_linea = linea['text'].upper()
            if "AMONESTACIONES - TARJETAS AMARILLAS" in texto_linea:
                y_amarillas = linea['top']
            elif "EXPULSIONES - TARJETAS ROJAS" in texto_linea:
                y_rojas = linea['top']
            elif "AMONESTACIONES DE CUERPO" in texto_linea:
                y_tecnico = linea['top']
                break # Ya no necesitamos leer más abajo

        # Si no hubo rojas en el partido, la caja de amarillas va hasta el cuerpo técnico
        limite_amarillas = y_rojas if y_rojas > 0 else y_tecnico

        # 2. DEFINIR CAJAS DELIMITADORAS
        caja_amarillas = (0, y_amarillas, pagina.width, limite_amarillas)
        caja_rojas = (0, y_rojas, pagina.width, y_tecnico) if y_rojas > 0 else None

        # 3. FUNCIÓN DE LIMPIEZA DE FILAS 
        def procesar_tabla(tabla_cruda, tipo_tarjeta):
            if not tabla_cruda: return []
            filas_procesadas = []
            
            # Evaluamos TODAS las filas, pero aplicamos filtros inteligentes
            for fila in tabla_cruda:
                if not fila or not fila[0]: continue
                
                tiempo_raw = str(fila[0]).replace('\n', '').replace("'", "").strip()
                
                # --- FILTRO ANTIBASURA ---
                # Si la celda contiene la palabra "Tiempo" o títulos de la tabla, la ignoramos
                if "TIEMPO" in tiempo_raw.upper() or "AMONESTACIONES" in str(fila[0]).upper() or "EXPULSIONES" in str(fila[0]).upper():
                    continue
                    
                # Si por error de lectura la fila tiene menos de 5 columnas, la ignoramos
                if len(fila) < 5:
                    continue

                dorsal_raw = str(fila[1]).replace('\n', '').strip()
                equipo = str(fila[2]).replace('\n', ' ').strip()
                jugador = str(fila[3]).replace('\n', ' ').strip()
                motivo = str(fila[4]).replace('\n', ' ').strip()
                
                # --- REGLA ESTRICTA: 2 APELLIDOS + 2 NOMBRES ---
                if ',' in jugador:
                    apellidos_str, nombres_str = jugador.split(',', 1)
                    
                    apellidos = apellidos_str.strip().split()[:2]
                    nombres = nombres_str.strip().split()[:2]
                    
                    jugador_limpio = f"{' '.join(apellidos)} {' '.join(nombres)}"
                else:
                    jugador_limpio = " ".join(jugador.split()[:4])

                # Ensamblaje del diccionario
                tarjeta_dict = {
                    "equipo": equipo,
                    "tipo_tarjeta": tipo_tarjeta,
                    "minuto": int(tiempo_raw) if tiempo_raw.isdigit() else tiempo_raw,
                    "dorsal": int(dorsal_raw) if dorsal_raw.isdigit() else None,
                    "jugador": jugador_limpio,
                    "motivo": motivo
                }
                
                filas_procesadas.append(tarjeta_dict)
                
            return filas_procesadas

        # 4. EXTRAER TABLAS DE LOS RECORTES Y PROCESAR
        if caja_amarillas:
            tabla_amarilla_cruda = pagina.crop(caja_amarillas).extract_table()
            resultados_disciplina.extend(procesar_tabla(tabla_amarilla_cruda, "AMARILLA"))

        if caja_rojas:
            tabla_roja_cruda = pagina.crop(caja_rojas).extract_table()
            resultados_disciplina.extend(procesar_tabla(tabla_roja_cruda, "ROJA"))

    return resultados_disciplina


########################
#EJECUCIÓN DE LOS DATOS

#INDICAMOS LA RUTA DEL ARCHIVO PDF A PROCESAR.
ruta_pdf = pathlib.Path("../datos/raw/TA-2025/TA-F10-01.05.25.pdf") #Definir la ruta del archivo PDF a procesar.

#SE CREA EL DICCIONARIO QUE ALMACENA LOS DATOS DEL PARTIDO
partido={}


#OBTENER LOS DATOS DEL ENCABEZADO DEL PDF Y AGREGARLA AL DICCIONARIO DEL PARTIDO
partido = extraer_datos_partido(ruta_pdf)
print("\n--- ENCABEZADO EXTRAÍDO ---")
for clave, valor in partido.items():
    print(f"{clave}: {valor}")


#OBTENER LA ALINEACION DEL EQUIPO LOCAL Y AGREGARLA AL DICCIONARIO DEL PARTIDO
alineacion_local = extraer_equipo_a(ruta_pdf, partido.get('equipo_local'))

print(f"\n--- ALINEACION LOCAL: {partido.get('equipo_local')} ---")
print("Lista - Dorsal - Apellidos y Nombres - Titular/Suplente - Arquero/Capitán")
for datos in alineacion_local:
    print(f"[{datos['numero_lista']}] - {datos['dorsal']} - {datos['nombre']} - {datos['titular_suplente']} - {datos['arquero_capitan']}") 

partido['alineacion_local'] = alineacion_local #Se agrega al Diccionario.


#OBTENER LA ALINEACION DEL EQUIPO VISITANTE Y AGREGARLA AL DICCIONARIO DEL PARTIDO
alineacion_visitante = extraer_equipo_b(ruta_pdf, partido.get('equipo_visitante'))

print(f"\n--- ALINEACION VISITANTE: {partido.get('equipo_visitante')} ---")
print("Lista - Dorsal - Apellidos y Nombres - Titular/Suplente - Arquero/Capitán")
for datos in alineacion_visitante:
    print(f"[{datos['numero_lista']}] - {datos['dorsal']} - {datos['nombre']} - {datos['titular_suplente']} - {datos['arquero_capitan']}") 

partido['alineacion_visitante'] = alineacion_visitante #Se agrega al Diccionario.


#OBTENER LAS SUSTITUCIONES Y AGREGARLA AL DICCIONARIO DEL PARTIDO
sustituciones = extraer_sustituciones(ruta_pdf)
print("\n--- SUSTITUCIONES DEL PARTIDO ---")
print("Equipo - Tiempo - Dorsal Entrante - Nombre Entrante -> Dorsal Saliente - Nombre Saliente")
for jugador in sustituciones:
    print(f"{jugador['equipo']} - {jugador['minuto']}' - {jugador['dorsal_entra']} - {jugador['jugador_entra']} -> {jugador['dorsal_sale']} - {jugador['jugador_sale']}") 

partido['sustituciones'] = sustituciones #Se agrega al Diccionario.


#OBTENER LOS GOLEADORES Y AGREGARLOS AL DICCIONARIO DEL PARTIDO
goleadores = extraer_goles(ruta_pdf)
print("\n--- GOLEADORES DEL PARTIDO ---")
print("Equipo - Minuto - Dorsal - Nombre del Goleador - Tipo de Gol (JUGADA/PENAL/AUTOGOL) - Condición (LOCAL/VISITANTE)")
for jugador in goleadores:
    print(f"{jugador['equipo']} - {jugador['minuto']}' - {jugador['dorsal']} - {jugador['jugador']} - {jugador['tipo_gol']} - {jugador['condicion']}") 

partido['goleadores'] = goleadores #Se agrega al Diccionario.


#OBTENER LOS AMONESTADOS Y EXPULSADOS Y AGREGARLOS AL DICCIONARIO DEL PARTIDO
tarjetas = extraer_tarjetas(ruta_pdf)
print("\n--- AMONESTADOS Y EXPULSADOS DEL PARTIDO ---")
print("Equipo - Tipo de Tarjeta - Tiempo - Dorsal - Nombre del Jugador - Motivo")
for jugador in tarjetas:
    print(f"{jugador['equipo']} - {jugador['tipo_tarjeta']} - {jugador['minuto']}' - {jugador['dorsal']} - {jugador['jugador']} - {jugador['motivo']}") 

partido['amonestados_expulsados'] = tarjetas #Se agrega al Diccionario.



--- ENCABEZADO EXTRAÍDO ---
liga: PROMESAS
municipio: CARONÍ
categoria: SUB-15
torneo: TA 2025
grupo: B
equipo_local: ACADEMIA DEPORTIVA EMANUEL
equipo_visitante: ACADEMIA M.F.C
goles_local: 0
goles_visitante: 1
goles_1T_local: 0
goles_1T_visitante: 1
fecha: 01/05/2025
hora: 11:15
estadio: Paratepuy
ciudad: Puerto Ordaz
pais: Venezuela

--- ALINEACION LOCAL: ACADEMIA DEPORTIVA EMANUEL ---
Lista - Dorsal - Apellidos y Nombres - Titular/Suplente - Arquero/Capitán
[1] - 23 - Mocco Espinoza Mathias Ezer - T - A
[2] - 4 - Parra Fazio Ricardo Javier - T - 
[3] - 6 - Figuera Diaz Maykol Jose - T - C
[4] - 12 - Rivas Cotua Luis Rafael - T - 
[5] - 30 - Gomez Sanchez Abram David - T - 
[6] - 36 - Bautista Aguilera Julio Cesar - T - 
[7] - 42 - Cedeño Ramirez Neomar Jesus - T - 
[8] - 43 - Osuna Parra Leonel Alejandro - T - 
[9] - 46 - Ramirez Ascanio Flavio Manuel - T - 
[10] - 48 - Alvarez Montalto Jacob Sebastian - T - 

--- ALINEACION VISITANTE: ACADEMIA M.F.C ---
Lista - Dorsal - Apellidos

In [4]:
#Visualización final del Diccionario estructurado con toda la información del Partido
print("\n--- DICCIONARIO FINAL DEL PARTIDO ---")
partido


--- DICCIONARIO FINAL DEL PARTIDO ---


{'liga': 'PROMESAS',
 'municipio': 'CARONÍ',
 'categoria': 'SUB-15',
 'torneo': 'TA 2025',
 'grupo': 'B',
 'equipo_local': 'ACADEMIA DEPORTIVA EMANUEL',
 'equipo_visitante': 'ACADEMIA M.F.C',
 'goles_local': 0,
 'goles_visitante': 1,
 'goles_1T_local': 0,
 'goles_1T_visitante': 1,
 'fecha': '01/05/2025',
 'hora': '11:15',
 'estadio': 'Paratepuy',
 'ciudad': 'Puerto Ordaz',
 'pais': 'Venezuela',
 'alineacion_local': [{'equipo': 'ACADEMIA DEPORTIVA EMANUEL',
   'numero_lista': 1,
   'dorsal': 23,
   'nombre': 'Mocco Espinoza Mathias Ezer',
   'titular_suplente': 'T',
   'arquero_capitan': 'A'},
  {'equipo': 'ACADEMIA DEPORTIVA EMANUEL',
   'numero_lista': 2,
   'dorsal': 4,
   'nombre': 'Parra Fazio Ricardo Javier',
   'titular_suplente': 'T',
   'arquero_capitan': ''},
  {'equipo': 'ACADEMIA DEPORTIVA EMANUEL',
   'numero_lista': 3,
   'dorsal': 6,
   'nombre': 'Figuera Diaz Maykol Jose',
   'titular_suplente': 'T',
   'arquero_capitan': 'C'},
  {'equipo': 'ACADEMIA DEPORTIVA EMANUEL',


#### Convertir Diccionario de Datos Extraidos en un DataFrame - Datos del Partido

In [5]:
#Crear una lista con el diccionario de Datos del Partido.

datos_partido = {
    "liga": partido.get('liga'),
    "municipio": partido.get('municipio'),
    "categoria": partido.get('categoria'),
    "torneo": partido.get('torneo'),
    "grupo": partido.get('grupo'),
    "equipo_local": partido.get('equipo_local'),
    "equipo_visitante": partido.get('equipo_visitante'),
    "goles_local": partido.get('goles_local'),
    "goles_visitante": partido.get('goles_visitante'),
    "goles_1T_local": partido.get('goles_1T_local'),
    "goles_1T_visitante": partido.get('goles_1T_visitante'),
    "fecha_partido": partido.get('fecha'),
    "hora_partido": partido.get('hora'),    
    "estadio": partido.get('estadio'),
    "ciudad": partido.get('ciudad'),
    "pais": partido.get('pais')
}

#Crear un DataFrame a partir de una lista con el diccionario de datos del partido.
df_partido = pd.DataFrame([datos_partido]) 

#Mostrar las primeras filas del DataFrame para verificar que los datos se han cargado correctamente.
df_partido.head()

,liga,municipio,categoria,torneo,grupo,equipo_local,equipo_visitante,goles_local,goles_visitante,goles_1T_local,goles_1T_visitante,fecha_partido,hora_partido,estadio,ciudad,pais
0,PROMESAS,CARONÍ,SUB-15,TA 2025,B,ACADEMIA DEPORTIVA EMANUEL,ACADEMIA M.F.C,0,1,0,1,01/05/2025,11:15,Paratepuy,Puerto Ordaz,Venezuela


#### DataFrames con las Alineaciones, Sustituciones, Goleadores y Amonestados

In [6]:
#Crear un DataFrame a partir de la alineación del equipo local.
df_alineacion_local = pd.DataFrame(partido.get('alineacion_local'))
df_alineacion_local.insert(0, 'fecha_partido', partido.get('fecha')) #Agregar la fecha del partido al principio del DataFrame de alineación local
print(f"\n--- DataFrame de Alineación Local ---")
display(df_alineacion_local) #Mostrar las filas del DataFrame para verificar que los datos se han cargado correctamente.

#Crear un DataFrame a partir de la alineación del equipo visitante.
df_alineacion_visitante = pd.DataFrame(partido.get('alineacion_visitante'))
df_alineacion_visitante.insert(0, 'fecha_partido', partido.get('fecha')) #Agregar la fecha del partido a cada jugador de la alineación visitante
print(f"\n--- DataFrame de Alineación Visitante ---")
display(df_alineacion_visitante) #Mostrar las filas del DataFrame para verificar que los datos se han cargado correctamente.

#Combinar ambos DataFrames de alineación en uno solo, agregando una columna que indique si el jugador es del equipo local o visitante.
df_alineacion_local['tipo_equipo'] = 'LOCAL'
df_alineacion_visitante['tipo_equipo'] = 'VISITANTE'
df_alineacion_completa = pd.concat([df_alineacion_local, df_alineacion_visitante], ignore_index=True)   
print(f"\n--- DataFrame de Alineación Completa ---")
display(df_alineacion_completa) #Mostrar las filas del DataFrame combinado para verificar que los datos se han cargado correctamente.

#Crear un DataFrame a partir de las sustituciones del partido.
df_sustituciones = pd.DataFrame(partido.get('sustituciones'))   
df_sustituciones.insert(0, 'fecha_partido', partido.get('fecha')) #Agregamos la fecha del partido a cada sustitución
print(f"\n--- DataFrame de Sustituciones ---")
display(df_sustituciones) #Mostrar las filas del DataFrame para verificar que los datos se han cargado correctamente.

#Crear un DataFrame a partir de los goleadores del partido.
df_goleadores = pd.DataFrame(partido.get('goleadores'))
df_goleadores.insert(0, 'fecha_partido', partido.get('fecha')) #Agregamos la fecha del partido a cada goleador
print(f"\n--- DataFrame de Goleadores ---")
display(df_goleadores) #Mostrar las filas del DataFrame para verificar que los datos se han cargado correctamente.


#Crear un DataFrame a partir de los amonestados y expulsados del partido.
df_disciplina = pd.DataFrame(partido.get('amonestados_expulsados'))
df_disciplina.insert(0, 'fecha_partido', partido.get('fecha')) #Agregamos la fecha del partido a cada jugador amonestado o expulsado
print(f"\n--- DataFrame de Disciplina ---")
display(df_disciplina) #Mostrar las filas del DataFrame para verificar que los datos se han cargado correctamente.  




--- DataFrame de Alineación Local ---


,fecha_partido,equipo,numero_lista,dorsal,nombre,titular_suplente,arquero_capitan
0,01/05/2025,ACADEMIA DEPORTIVA EMANUEL,1,23,Mocco Espinoza Mathias Ezer,T,A
1,01/05/2025,ACADEMIA DEPORTIVA EMANUEL,2,4,Parra Fazio Ricardo Javier,T,
2,01/05/2025,ACADEMIA DEPORTIVA EMANUEL,3,6,Figuera Diaz Maykol Jose,T,C
3,01/05/2025,ACADEMIA DEPORTIVA EMANUEL,4,12,Rivas Cotua Luis Rafael,T,
4,01/05/2025,ACADEMIA DEPORTIVA EMANUEL,5,30,Gomez Sanchez Abram David,T,
5,01/05/2025,ACADEMIA DEPORTIVA EMANUEL,6,36,Bautista Aguilera Julio Cesar,T,
6,01/05/2025,ACADEMIA DEPORTIVA EMANUEL,7,42,Cedeño Ramirez Neomar Jesus,T,
7,01/05/2025,ACADEMIA DEPORTIVA EMANUEL,8,43,Osuna Parra Leonel Alejandro,T,
8,01/05/2025,ACADEMIA DEPORTIVA EMANUEL,9,46,Ramirez Ascanio Flavio Manuel,T,
9,01/05/2025,ACADEMIA DEPORTIVA EMANUEL,10,48,Alvarez Montalto Jacob Sebastian,T,



--- DataFrame de Alineación Visitante ---


,fecha_partido,equipo,numero_lista,dorsal,nombre,titular_suplente,arquero_capitan
0,01/05/2025,ACADEMIA M.F.C,1,99,Rosales Gomez Moises Sebastian,T,A
1,01/05/2025,ACADEMIA M.F.C,2,5,Maita Salas Jeremias Jatniel,T,C
2,01/05/2025,ACADEMIA M.F.C,3,7,Adedigba Zapata Emmanuel Aderemi,T,
3,01/05/2025,ACADEMIA M.F.C,4,8,Rincones Ruiz Guillermo Alexander,T,
4,01/05/2025,ACADEMIA M.F.C,5,10,Sandoval Noguera Dionner Alejandro,T,
5,01/05/2025,ACADEMIA M.F.C,6,11,Sanchez Rada Emmanuel Abraham,T,
6,01/05/2025,ACADEMIA M.F.C,7,15,Arias Medina Angel Leonardo,T,
7,01/05/2025,ACADEMIA M.F.C,8,16,Carmona Loreto Ignacio David,T,
8,01/05/2025,ACADEMIA M.F.C,9,18,Diaz Villegas Samuel Alejandro,T,
9,01/05/2025,ACADEMIA M.F.C,10,19,Parra Bolivar Diego Alejandro,T,



--- DataFrame de Alineación Completa ---


...

,fecha_partido,equipo,minuto,dorsal_entra,jugador_entra,dorsal_sale,jugador_sale
0,01/05/2025,ACADEMIA M.F.C,36,26,Gutierrez Rangel Abraham David,18,Diaz Villegas Samuel Alejandro
1,01/05/2025,ACADEMIA M.F.C,45,25,Marcano Camacho Juan Marcos,16,Carmona Loreto Ignacio David
2,01/05/2025,ACADEMIA M.F.C,51,22,Suarez Echeverria Jeshua Alexander,10,Sandoval Noguera Dionner Alejandro



--- DataFrame de Goleadores ---


,fecha_partido,equipo,minuto,dorsal,jugador,tipo_gol,condicion
0,01/05/2025,ACADEMIA M.F.C,25,21,Marcano Alcala Nelson Javier,JUGADA,VISITANTE



--- DataFrame de Disciplina ---


,fecha_partido,equipo,tipo_tarjeta,minuto,dorsal,jugador,motivo
0,01/05/2025,ACADEMIA DEPORTIVA EMANUEL,AMARILLA,22,43,Osuna Parra Leonel Alejandro,Conducta antideportiva
1,01/05/2025,ACADEMIA M.F.C,AMARILLA,50,7,Adedigba Zapata Emmanuel Aderemi,Conducta antideportiva
2,01/05/2025,ACADEMIA M.F.C,AMARILLA,65,25,Marcano Camacho Juan Marcos,Infringir
3,01/05/2025,ACADEMIA DEPORTIVA EMANUEL,AMARILLA,68,43,Osuna Parra Leonel Alejandro,Conducta antideportiva
4,01/05/2025,ACADEMIA DEPORTIVA EMANUEL,ROJA,68,43,Osuna Parra Leonel Alejandro,Segunda tarjeta amarilla (Conducta antideporti...


#### Revisión de Valores en DataFrame Goleadores

In [7]:
#Mostrar la información del DataFrame de los Goleadores, incluyendo el número de filas, columnas, tipos de datos y valores nulos.
print(f"\n--- DataFrame de Goleadores ---")
df_goleadores.info()

#Se revisan los valores nulos por columna, para identificar posibles datos faltantes.
print("\nValores Nulos por Columna:")
Valores_Nulos= df_goleadores.isnull().sum()
print(Valores_Nulos)


--- DataFrame de Goleadores ---
<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   fecha_partido  1 non-null      str  
 1   equipo         1 non-null      str  
 2   minuto         1 non-null      int64
 3   dorsal         1 non-null      int64
 4   jugador        1 non-null      str  
 5   tipo_gol       1 non-null      str  
 6   condicion      1 non-null      str  
dtypes: int64(2), str(5)
memory usage: 188.0 bytes

Valores Nulos por Columna:
fecha_partido    0
equipo           0
minuto           0
dorsal           0
jugador          0
tipo_gol         0
condicion        0
dtype: int64


#### Normalizar Strings a Mayusculas

In [10]:
#Se copia el dataframe, para realizar las transformaciones necesarias sin afectar el original.
#Se convierten los strings en mayúsculas, para estandarizar la información
df_goleadores_correcciones = df_goleadores.copy()
df_goleadores_correcciones['equipo'] = df_goleadores_correcciones['equipo'].str.upper()
df_goleadores_correcciones['jugador'] = df_goleadores_correcciones['jugador'].str.upper()
df_goleadores_correcciones['tipo_gol'] = df_goleadores_correcciones['tipo_gol'].str.upper()

#Se muestran las primeras filas del dataframe, para verificar su contenido.
print(f"\n--- DataFrame de Goleadores en Mayúsculas ---")
display(df_goleadores_correcciones)


--- DataFrame de Goleadores en Mayúsculas ---


,fecha_partido,equipo,minuto,dorsal,jugador,tipo_gol,condicion
0,01/05/2025,ACADEMIA M.F.C,25,21,MARCANO ALCALA NELSON JAVIER,JUGADA,VISITANTE


#### Corregir Tipos de Datos (Campo String a Fecha)

In [11]:
#Verificar tipo de Datos en Campo Fecha
print("\nTipo de Datos en Campo fecha: ", df_goleadores_correcciones['fecha_partido'].dtype)

#Convertir Campo Fecha de String a DateTime.
df_goleadores_correcciones['fecha_partido'] = pd.to_datetime(df_goleadores_correcciones['fecha_partido'], format='%d/%m/%Y')

print("Tipo de Datos en Campo fecha, luego de la conversión: ", df_goleadores_correcciones['fecha_partido'].dtype)

#Se muestran las filas del dataset, para verificar su contenido.
display(df_goleadores_correcciones)

#Información general del dataset, para verificar las correcciones realizadas.
print(f"\n--- DataFrame de Datos del Partido con correcciones de Fecha y Hora ---")
df_goleadores_correcciones.info()


Tipo de Datos en Campo fecha:  str
Tipo de Datos en Campo fecha, luego de la conversión:  datetime64[us]


,fecha_partido,equipo,minuto,dorsal,jugador,tipo_gol,condicion
0,2025-05-01,ACADEMIA M.F.C,25,21,MARCANO ALCALA NELSON JAVIER,JUGADA,VISITANTE



--- DataFrame de Datos del Partido con correcciones de Fecha y Hora ---
<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   fecha_partido  1 non-null      datetime64[us]
 1   equipo         1 non-null      str           
 2   minuto         1 non-null      int64         
 3   dorsal         1 non-null      int64         
 4   jugador        1 non-null      str           
 5   tipo_gol       1 non-null      str           
 6   condicion      1 non-null      str           
dtypes: datetime64[us](1), int64(2), str(4)
memory usage: 188.0 bytes


### Procesar Carpeta de Planillas PDFs - Liga Promesas, Sub15, TA 2025, Grupo B

#### Extraer a los Goleadores

In [8]:
# Función para procesar una carpeta entera de PDFs con los goleadores
def procesar_goleadores_torneo_pdfs(carpeta):
    
    direccion_pdfs = Path(carpeta)   
    archivos_pdfs = list(direccion_pdfs.glob('*.pdf'))
    
    print(f"📁 Se encuentran {len(archivos_pdfs)} PDFs en la carpeta '{direccion_pdfs}'")
    
    goles_totales = [] 
    errores  = []
    
    for pdf_path in sorted(archivos_pdfs):
        try:
            datos_goleadores_torneo = extraer_goles(str(pdf_path)) 
            
            # 1. Verificamos que sea una LISTA y que no esté vacía
            if isinstance(datos_goleadores_torneo, list) and len(datos_goleadores_torneo) > 0:
                
                # 2. Iteramos sobre cada gol dentro de la lista de ese partido
                for gol in datos_goleadores_torneo:
                    fila = {
                        "archivo": pdf_path.name,                    
                        "equipo": gol.get('equipo'),
                        "minuto": gol.get('minuto'),
                        "dorsal": gol.get('dorsal'),
                        "jugador": gol.get('jugador'), 
                        "tipo_gol": gol.get('tipo_gol'),
                        "condicion": gol.get('condicion')              
                    }
                    goles_totales.append(fila)
                    print(f"  ✅ {pdf_path.name} → {fila['equipo']} - MIN: {fila['minuto']} - DORSAL: {fila['dorsal']} - {fila['jugador']}")
            else:
                print(f"  ⏭️ {pdf_path.name} → Sin goles registrados (0-0) o Formato inválido")
                
        except Exception as e:
            errores.append(pdf_path.name)
            print(f"  ❌ Error en {pdf_path.name}: {e}")
    
    if errores:
        print(f"\n⚠️ {len(errores)} archivos con error: {errores}")
    
    if not goles_totales:
        return pd.DataFrame()
    
    df = pd.DataFrame(goles_totales)
    print(f"\n📊 Resultado: {len(df)} goles procesados correctamente")
    return df

# Procesar los PDFs
df_goleadores_torneo_apertura = procesar_goleadores_torneo_pdfs('../datos/raw/TA-2025/')

# Mostrar las filas del DataFrame
print(f"\n--- DataFrame de los Goleadores Sub-15 del Torneo Apertura 2025 ---")
display(df_goleadores_torneo_apertura)

# Exportar el DataFrame a un archivo CSV para su análisis posterior
df_goleadores_torneo_apertura.to_csv('../datos/processed/TablaGeneral_Goleadores_TA25_GrupoB.csv', index=False)
print("\n✅ DataFrame exportado a '../datos/processed/TablaGeneral_Goleadores_TA25_GrupoB.csv'")


📁 Se encuentran 12 PDFs en la carpeta '..\datos\raw\TA-2025'
  ✅ TA-F01-12.04.25.pdf → CORE SPORT - MIN: 8 - DORSAL: 23 - Rojas Mendez Cristian Jesus
  ✅ TA-F01-12.04.25.pdf → CORE SPORT - MIN: 24 - DORSAL: 23 - Rojas Mendez Cristian Jesus
  ✅ TA-F01-12.04.25.pdf → CORE SPORT - MIN: 35 - DORSAL: 16 - Cavadia Oca Luis Mario
  ✅ TA-F01-12.04.25.pdf → CORE SPORT - MIN: 43 - DORSAL: 16 - Cavadia Oca Luis Mario
  ✅ TA-F02-21.03.25.pdf → S LUGO FC - MIN: 65 - DORSAL: 8 - Valdez Roca Gabriel Alexander
  ✅ TA-F02-21.03.25.pdf → ACADEMIA M.F.C - MIN: 16 - DORSAL: 37 - Gutierrez Bolivar Mathias Eduardo
  ✅ TA-F02-21.03.25.pdf → ACADEMIA M.F.C - MIN: 35 - DORSAL: 37 - Gutierrez Bolivar Mathias Eduardo
  ✅ TA-F03-25.03.25.pdf → ACADEMIA M.F.C - MIN: 10 - DORSAL: 8 - Rincones Ruiz Guillermo Alexander
  ✅ TA-F03-25.03.25.pdf → ACADEMIA M.F.C - MIN: 11 - DORSAL: 37 - Gutierrez Bolivar Mathias Eduardo
  ✅ TA-F03-25.03.25.pdf → ACADEMIA M.F.C - MIN: 26 - DORSAL: 37 - Gutierrez Bolivar Mathias Eduardo
 

,archivo,equipo,minuto,dorsal,jugador,tipo_gol,condicion
0,TA-F01-12.04.25.pdf,CORE SPORT,8,23,Rojas Mendez Cristian Jesus,JUGADA,VISITANTE
1,TA-F01-12.04.25.pdf,CORE SPORT,24,23,Rojas Mendez Cristian Jesus,JUGADA,VISITANTE
2,TA-F01-12.04.25.pdf,CORE SPORT,35,16,Cavadia Oca Luis Mario,JUGADA,VISITANTE
3,TA-F01-12.04.25.pdf,CORE SPORT,43,16,Cavadia Oca Luis Mario,JUGADA,VISITANTE
4,TA-F02-21.03.25.pdf,S LUGO FC,65,8,Valdez Roca Gabriel Alexander,JUGADA,LOCAL
5,TA-F02-21.03.25.pdf,ACADEMIA M.F.C,16,37,Gutierrez Bolivar Mathias Eduardo,JUGADA,VISITANTE
6,TA-F02-21.03.25.pdf,ACADEMIA M.F.C,35,37,Gutierrez Bolivar Mathias Eduardo,JUGADA,VISITANTE
7,TA-F03-25.03.25.pdf,ACADEMIA M.F.C,10,8,Rincones Ruiz Guillermo Alexander,JUGADA,LOCAL
8,TA-F03-25.03.25.pdf,ACADEMIA M.F.C,11,37,Gutierrez Bolivar Mathias Eduardo,JUGADA,LOCAL
9,TA-F03-25.03.25.pdf,ACADEMIA M.F.C,26,37,Gutierrez Bolivar Mathias Eduardo,JUGADA,LOCAL



✅ DataFrame exportado a '../datos/processed/TablaGeneral_Goleadores_TA25_GrupoB.csv'


### Enriquecer DataFrame de los Goleadores - Liga Promesas, Sub15, TA 2025, Grupo B

#### Tabla de Goleadores Integral

In [9]:
# Crear un ranking avanzado desglosando el tipo de anotación
df_goleadores_integral = df_goleadores_torneo_apertura.groupby(['jugador']).agg(
    equipo=('equipo', 'first'),
    total_goles=('jugador', 'count'),
    goles_jugada=('tipo_gol', lambda x: (x == 'JUGADA').sum()),
    goles_penal=('tipo_gol', lambda x: (x == 'PEN').sum()),
    goles_autogol=('tipo_gol', lambda x: (x == 'AG').sum()) 
).reset_index()

# Ordenar y ajustar índice
df_goleadores_integral = df_goleadores_integral.sort_values(by='total_goles', ascending=False).reset_index(drop=True)
df_goleadores_integral.index = df_goleadores_integral.index + 1

#Convertir los strings en mayúsculas, para estandarizar la información
df_goleadores_integral['jugador'] = df_goleadores_integral['jugador'].str.upper()

#Mostrar el ranking de goleadores detallado, con el total de goles y el desglose por tipo de anotación.
print("\n📊 TABLA DE GOLEADORES SUB-15 - TORNEO APERTURA 2025 - GRUPO B")
display(df_goleadores_integral)


📊 TABLA DE GOLEADORES SUB-15 - TORNEO APERTURA 2025 - GRUPO B


,jugador,equipo,total_goles,goles_jugada,goles_penal,goles_autogol
1,ADEDIGBA ZAPATA EMMANUEL ADEREMI,ACADEMIA M.F.C,7,7,0,0
2,GUTIERREZ BOLIVAR MATHIAS EDUARDO,ACADEMIA M.F.C,6,6,0,0
3,VALDEZ ROCA GABRIEL ALEXANDER,S LUGO FC,3,3,0,0
4,MARCANO ALCALA NELSON JAVIER,ACADEMIA M.F.C,2,2,0,0
5,GONZALEZ MARCANO GUILLERMO ALEJANDRO,ACADEMIA SUR AEROPUERTO FC,2,2,0,0
6,CAVADIA OCA LUIS MARIO,CORE SPORT,2,2,0,0
7,SEBASTIAN SHAMA HERNANDEZ SALAZAR,ESCUELA DE FUTBOL MGD,2,2,0,0
8,ROJAS MENDEZ CRISTIAN JESUS,CORE SPORT,2,2,0,0
9,BACA VELASQUEZ ANGEL EZEQUIEL,ACADEMIA SUR AEROPUERTO FC,1,1,0,0
10,AGUILERA REYES DAVID ENRIQUE,ESCUELA DE FUTBOL MGD,1,1,0,0


#### Tablas de Goleadores Detalladas

In [10]:
# --- 1. Agrupamos por jugador y realizamos todos los cálculos en un solo paso ---

df_goleadores_detallado = df_goleadores_torneo_apertura.groupby('jugador').agg(
    equipo=('equipo', 'first'),
    total_goles=('jugador', 'count'),
    goles_local=('condicion', lambda x: (x == 'LOCAL').sum()),
    goles_visitante=('condicion', lambda x: (x == 'VISITANTE').sum()),
    jugada=('tipo_gol', lambda x: (x == 'JUGADA').sum()),
    penal=('tipo_gol', lambda x: (x == 'PEN').sum()),
    autogol=('tipo_gol', lambda x: (x == 'AG').sum())
).reset_index()

# Ordenar y estandarizar
df_goleadores_detallado = df_goleadores_detallado.sort_values(by='total_goles', ascending=False).reset_index(drop=True)
df_goleadores_detallado['jugador'] = df_goleadores_detallado['jugador'].str.upper()

# --- 2. AGREGAR FILA DE TOTALES ---
# Seleccionamos solo las columnas numéricas para sumar
columnas_numericas = ['total_goles', 'goles_local', 'goles_visitante', 'jugada', 'penal', 'autogol']
totales = df_goleadores_detallado[columnas_numericas].sum()

# Creamos la fila de total asignando etiquetas a las columnas no numéricas
fila_total = pd.Series(data=totales, index=columnas_numericas)
fila_total['jugador'] = 'TOTALES'
fila_total['equipo'] = '-' # Opcional: puedes dejarlo vacío o con un guion

# Añadimos la fila al final del DataFrame
df_final_con_totales = pd.concat([df_goleadores_detallado, fila_total.to_frame().T], ignore_index=True)

# Ajustar el índice para que el ranking empiece en 1 (y la fila total no tenga número de ranking)
df_final_con_totales.index = df_final_con_totales.index + 1

print("\n📊 TABLA DE GOLEADORES SUB-15 DETALLADA - TORNEO APERTURA 2025 - GRUPO B")
display(df_final_con_totales)


📊 TABLA DE GOLEADORES SUB-15 DETALLADA - TORNEO APERTURA 2025 - GRUPO B


,jugador,equipo,total_goles,goles_local,goles_visitante,jugada,penal,autogol
1,ADEDIGBA ZAPATA EMMANUEL ADEREMI,ACADEMIA M.F.C,7,5,2,7,0,0
2,GUTIERREZ BOLIVAR MATHIAS EDUARDO,ACADEMIA M.F.C,6,2,4,6,0,0
3,VALDEZ ROCA GABRIEL ALEXANDER,S LUGO FC,3,1,2,3,0,0
4,MARCANO ALCALA NELSON JAVIER,ACADEMIA M.F.C,2,0,2,2,0,0
5,GONZALEZ MARCANO GUILLERMO ALEJANDRO,ACADEMIA SUR AEROPUERTO FC,2,2,0,2,0,0
6,CAVADIA OCA LUIS MARIO,CORE SPORT,2,0,2,2,0,0
7,SEBASTIAN SHAMA HERNANDEZ SALAZAR,ESCUELA DE FUTBOL MGD,2,1,1,2,0,0
8,ROJAS MENDEZ CRISTIAN JESUS,CORE SPORT,2,0,2,2,0,0
9,BACA VELASQUEZ ANGEL EZEQUIEL,ACADEMIA SUR AEROPUERTO FC,1,1,0,1,0,0
10,AGUILERA REYES DAVID ENRIQUE,ESCUELA DE FUTBOL MGD,1,1,0,1,0,0


#### Tablas de Goleadores Detallada - ACADEMIA M.F.C

In [16]:
# --- VISTA FILTRADA ACADEMIA MFC CON SUS PROPIOS TOTALES ---
df_mfc = df_goleadores_detallado[df_goleadores_detallado['equipo'].str.contains('ACADEMIA M.F.C', case=False, na=False)].copy()

if not df_mfc.empty:
    sumas_mfc = df_mfc[columnas_numericas].sum()
    fila_total_mfc = pd.Series(data=sumas_mfc, index=columnas_numericas)
    fila_total_mfc['jugador'] = 'TOTAL CANTERA MFC'
    fila_total_mfc['equipo'] = 'ACADEMIA M.F.C'
    
    df_mfc_final = pd.concat([df_mfc, fila_total_mfc.to_frame().T], ignore_index=True)
    df_mfc_final.index = df_mfc_final.index + 1
    
    print("\n⚽ DESEMPEÑO INDIVIDUAL: GOLEADORES SUB-15 CANTERA ACADEMIA MFC - TORNEO APERTURA 2025 - GRUPO B")
    display(df_mfc_final)


⚽ DESEMPEÑO INDIVIDUAL: GOLEADORES SUB-15 CANTERA ACADEMIA MFC - TORNEO APERTURA 2025 - GRUPO B


,jugador,equipo,total_goles,goles_local,goles_visitante,jugada,penal,autogol
1,ADEDIGBA ZAPATA EMMANUEL ADEREMI,ACADEMIA M.F.C,7,5,2,7,0,0
2,GUTIERREZ BOLIVAR MATHIAS EDUARDO,ACADEMIA M.F.C,6,2,4,6,0,0
3,MARCANO ALCALA NELSON JAVIER,ACADEMIA M.F.C,2,0,2,2,0,0
4,DIAZ VILLEGAS SAMUEL ALEJANDRO,ACADEMIA M.F.C,1,1,0,1,0,0
5,MARCANO CAMACHO JUAN MARCOS,ACADEMIA M.F.C,1,1,0,0,1,0
6,RINCONES RUIZ GUILLERMO ALEXANDER,ACADEMIA M.F.C,1,1,0,1,0,0
7,TOTAL CANTERA MFC,ACADEMIA M.F.C,18,10,8,17,1,0


### Estadisticas Descriptivas - Data Frame Goleadores Sub15 - Academia M.F.C

In [17]:
# Mostrar las Estadísticas Descriptivas del DataFrame, 

print(f"\n--- Estadísticas Descriptivas del DataFrame de GOLEADORES SUB-15 - CANTERA ACADEMIA MFC - TORNEO APERTURA 2025 - GRUPO B ---")
df_mfc.describe()


--- Estadísticas Descriptivas del DataFrame de GOLEADORES SUB-15 - CANTERA ACADEMIA MFC - TORNEO APERTURA 2025 - GRUPO B ---


,total_goles,goles_local,goles_visitante,jugada,penal,autogol
count,6.00000,6.000000,6.000000,6.000000,6.000000,6.0
mean,3.00000,1.666667,1.333333,2.833333,0.166667,0.0
std,2.75681,1.751190,1.632993,2.926887,0.408248,0.0
min,1.00000,0.000000,0.000000,0.000000,0.000000,0.0
25%,1.00000,1.000000,0.000000,1.000000,0.000000,0.0
50%,1.50000,1.000000,1.000000,1.500000,0.000000,0.0
75%,5.00000,1.750000,2.000000,5.000000,0.000000,0.0
max,7.00000,5.000000,4.000000,7.000000,1.000000,0.0


### Exportar Tabla de Goleadores Liga Promesas, Sub15, TA 2025, Grupo B, a archivo CSV

In [11]:
#Exportar el dataset a un nuevo archivo CSV, para su uso en análisis posteriores.
df_final_con_totales.to_csv('../datos/processed/Goleadores_TA25_GrupoB_clean.csv', index=False)
print("\n✅ Dataset exportado correctamente a '../datos/processed/Goleadores_TA25_GrupoB_clean.csv'")


✅ Dataset exportado correctamente a '../datos/processed/Goleadores_TA25_GrupoB_clean.csv'
